# 06 · Particion y fuga de datos

> **Objetivo:** definir las cohortes A (entrenamiento) y B (prueba) con
> corte temporal, y demostrar las dos formas en que la **fuga de datos**
> rompe silenciosamente un modelo honesto.

## Que vamos a hacer

1. Construir las cohortes A (corte 2011-06-01) y B (corte 2011-09-01)
   con la funcion `cohorte(ventas, cancelaciones, corte, horizonte=90)`.
2. Mostrar la **fuga en la variable `recencia`**: si `recencia` se
   calcula con compras posteriores al corte, el AUC de la cohorte B
   salta de **0,795** a **0,955**.
3. Mostrar la **fuga en el preprocesamiento**: si el `StandardScaler`
   se ajusta con A ∪ B, la media de `recencia` cambia de **168,7** a
   **186,9** dias (filtracion de la distribucion de B hacia A).

## Reglas (Tabla 8 del OVA)

- `fit` del pipeline solo con la cohorte de entrenamiento.
- Variables construidas solo con datos anteriores al corte.
- Semilla fija y orden fijo de filas (`.sort("Customer ID")` dentro de
  `cohorte`); sin eso, el bosque no es reproducible.

In [1]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from segmentacion.config import FECHA_CORTE_A, FECHA_CORTE_B, HORIZONTE
from segmentacion.data.cargar import leer_parquet
from segmentacion.data.limpieza import preparar_ventas
from segmentacion.features.cohortes import cohorte

RAW = PROJECT_ROOT / "data" / "raw" / "online_retail_II.parquet"
assert RAW.exists(), "Ejecuta `uv run python scripts/descargar_datos.py` primero."

ventas, canc = preparar_ventas(leer_parquet(RAW), log=False)
A = cohorte(ventas, canc, FECHA_CORTE_A, horizonte=HORIZONTE)
B = cohorte(ventas, canc, FECHA_CORTE_B, horizonte=HORIZONTE)
print(f"A: {A.height} clientes | positivos {A['recompra'].sum()} ({A['recompra'].mean():.1%})")
print(f"B: {B.height} clientes | positivos {B['recompra'].sum()} ({B['recompra'].mean():.1%})")

A: 4933 clientes | positivos 1621 (32.9%)
B: 5249 clientes | positivos 2241 (42.7%)


## 1. Que es la fuga temporal

La **fuga de datos** (data leakage) ocurre cuando una variable usada en
entrenamiento contiene informacion que no se podria tener en el momento
de la prediccion. Es facil de hacer sin darse cuenta: por ejemplo,
calcular la recencia como "dias desde la ultima compra" sin filtrar
por la fecha de corte.

En el repositorio del modulo tenemos el ejemplo generico en
`projects/d2-temporal-split/data_leakage_temporal.ipynb`. Aqui lo vemos
sobre el caso real de recompra.

## 2. Fuga en la variable `recencia`

`cohorte` incluye dos columnas:

- `recencia`: dias entre `max(InvoiceDate < corte)` y el corte. **Honesta.**
- `recencia_fuga`: dias entre `max(InvoiceDate)` (sin filtrar) y el corte.
  **Contaminada**: usa compras posteriores al corte.

Entrenamos una regresion logistica con cada una.

In [3]:
from segmentacion.evaluation.metricas import fuga_recencia
from segmentacion.models.entrenar import MODELOS, entrenar
from segmentacion.pipelines.preprocesamiento import CAT, NUM_LOG, NUM_STD, construir_preprocesador
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

cols = NUM_LOG + NUM_STD + CAT
XA = A.select(cols).to_pandas()
yA = A["recompra"].to_numpy()
XB = B.select(cols).to_pandas()
yB = B["recompra"].to_numpy()

# Honesto
cv = StratifiedKFold(5, shuffle=True, random_state=42)
pipe_h = Pipeline([("pre", construir_preprocesador()), ("mod", MODELOS["Regresión logística"])])
res_h = entrenar(pipe_h, XA, yA, cv=cv)
pipe_h.fit(XA, yA)
auc_h_B = roc_auc_score(yB, pipe_h.predict_proba(XB)[:, 1])
print(f"honesto: AUC CV (A) = {res_h['cv_auc']:.3f} | AUC (B) = {auc_h_B:.3f}")

# Fuga
leak = fuga_recencia(
    XA, yA, XB, yB,
    A["recencia_fuga"].to_numpy(), B["recencia_fuga"].to_numpy(),
    construir_preprocesador,
)
print(f"con fuga: AUC CV (A) = {leak['cv_auc']:.3f} | AUC (B) = {leak['test_auc']:.3f}")

honesto: AUC CV (A) = 0.817 | AUC (B) = 0.795
con fuga: AUC CV (A) = 0.881 | AUC (B) = 0.955


### Lectura

El AUC CV sube de **0,817 → 0,881** y el AUC en B salta a **0,955**.
Parece un modelo excelente, pero es un fraude: en produccion no
tendriamos la recencia que incluye el futuro. **La metrica alta no
siempre indica un buen modelo.** Siempre que un AUC parezca
sospechosamente alto, hay que inspeccionar las variables.

## 3. Fuga en el preprocesamiento

Si el `StandardScaler` ve A y B juntos, aprende la media de A ∪ B.
Esto no es una fuga tan dramatica (los modelos no colapsan), pero
rompe el supuesto de que el modelo nunca vio B.

In [4]:
from segmentacion.evaluation.metricas import fuga_escalado

esc = fuga_escalado(XA, XB, NUM_STD)
print(f"media de `recencia` ajustando con A ∪ B: {esc['media_recencia_todo']:.1f}")
print(f"media de `recencia` ajustando solo con A: {esc['media_recencia_train']:.1f}")

media de `recencia` ajustando con A ∪ B: 186.9
media de `recencia` ajustando solo con A: 168.7


La diferencia (**186,9 vs 168,7** dias) existe porque B es tres
meses posterior a A y los clientes "activos" tienen distribucion
distinta. La magnitud parece pequeña, pero el modelo aprende a
calibrarse contra la media y el `log1p` previo amplifica cualquier
sesgo. **Toda operacion que vea datos debe ocurrir despues de la
particion.** El `Pipeline` de scikit-learn lo garantiza por construccion.

## Preguntas de reflexion

1. Si el `recencia_fuga` esta en el DataFrame, ¿por que no borrarlo
   despues de construirlo? (Pista: queremos poder enseñar el error.)
2. ¿Que otra variable del dominio podria filtrar informacion del
   futuro sin que lo notemos? Piensa en el `tasa_devolucion` o el `pais`.
3. Si en U4 (app Streamlit) un usuario pregunta por un cliente nuevo,
   ¿que valores tienen las variables? ¿Como se calcula la recencia?

> **Siguiente paso:** `07_ingenieria_caracteristicas.ipynb` — abrir
> el `ColumnTransformer` y entender que aprende cada transformador.